# ChipFF Bench: thermal expansion on a free GPU (Kaggle)

Runs the GPU part of ChipFF Bench: constant-pressure MD in kUPS for every material x model. The CPU parts
(lattice constants, elastic tensors) are already inside the project under `results/raw/`.

**One-time setup (Kaggle)**
1. Verify your phone number (Settings → Phone verification). Kaggle needs this before notebooks can use the internet.
2. Datasets → New dataset → upload `chipff-bench.zip` (made by `scripts/make_kaggle_bundle.sh`). Name it `chipff-bench`, keep it **Private**.
3. Notebook: File → Import notebook → this file. Then in the right panel: Add input → your `chipff-bench` dataset;
   Session options → Accelerator **GPU P100** (best for this job; T4 also works), Internet **On**.
4. Optional, for UMA: Add-ons → Secrets → add `HF_TOKEN` (a Hugging Face read token whose account has UMA access).

**Run it unattended:** click **Save Version → Save & Run All (Commit)**. It runs in the background (you can close the tab),
up to ~12 h. When it finishes, open the version → **Output** → download `chipff_thermal_results.zip`, and unzip it in the
project folder on your computer (it only contains `results/raw/*/thermal*.json`).

## 1. Settings

In [ ]:
MODELS = ["mace-mpa-0", "orb-v3", "uma-s-1p2"]   # MatterSim is run without kUPS and has no thermal step
RUN_PILOT = True          # 2 models x 4 materials, 20 ps: checks the method and prints timings first
RUN_FULL = True
TIME_BUDGET_H = 11.0      # stop starting new models after this (Kaggle sessions end at 12 h)
GITHUB_REPO = ""          # optional instead of the dataset, e.g. "https://github.com/anushajain2/chipff-bench"

## 2. Get the code and install (Python 3.12 via uv, since kUPS needs Python ≥ 3.12)

In [ ]:
import glob, os, shutil, subprocess, time
T0 = time.time()
!nvidia-smi --query-gpu=name,memory.total --format=csv
WORK = "/kaggle/working/chipff-bench" if os.path.exists("/kaggle") else os.path.abspath("chipff-bench")
if not os.path.exists(WORK):
    if GITHUB_REPO:
        subprocess.run(["git", "clone", "-q", GITHUB_REPO, WORK], check=True)
    else:
        hits = glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)
        assert hits, "Attach the chipff-bench dataset (Add input) or set GITHUB_REPO"
        shutil.copytree(os.path.dirname(hits[0]), WORK)
%cd {WORK}
!pip -q install uv
!uv venv -q --python 3.12 .venv
!uv pip install -q --python .venv/bin/python -e ".[uma,cuda]"
!.venv/bin/python -c "import jax; print('JAX devices:', jax.devices())"

## 3. Hugging Face token (only needed for UMA)

In [ ]:
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        pass
if token:
    os.environ["HF_TOKEN"] = token
ok = token and subprocess.run([".venv/bin/python", "-c",
    "from huggingface_hub import hf_hub_download as d; d('facebook/UMA','checkpoints/uma-s-1p2.pt')"]).returncode == 0
if not ok:
    MODELS = [m for m in MODELS if not m.startswith("uma")]
    print("UMA not accessible here (no HF_TOKEN secret, or license not yet approved): skipping it.")
print("Models:", MODELS)

## 4. Pilot: method check + timings

In [ ]:
!.venv/bin/chipff plan
if RUN_PILOT:
    !.venv/bin/chipff thermal -m mace-mpa-0 orb-v3 --materials Si Cu MgO GaN --max-prod-ps 20 --tag thermal_pilot
    !.venv/bin/python scripts/thermal_status.py --tag thermal_pilot

## 5. Full run\nOne model at a time; results are saved after every batch. Re-running skips what is already done.

In [ ]:
if RUN_FULL:
    for m in MODELS:
        if (time.time() - T0) / 3600 > TIME_BUDGET_H:
            print("Time budget reached; not starting", m)
            break
        !.venv/bin/chipff thermal -m {m}
    !.venv/bin/python scripts/thermal_status.py --tag thermal

## 6. Package the results

In [ ]:
!cd {WORK} && zip -q -r /kaggle/working/chipff_thermal_results.zip results/raw/*/thermal*.json logs 2>/dev/null; ls -la /kaggle/working/*.zip